# 04_extract_text

Dry-run text extraction on the latest inventory output. This notebook stays conservative: text-like files, PDF text extraction, DOCX paragraphs/tables, and XLSX sheet previews. No OCR yet.


In [1]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.inventory import ensure_inventory_schema
from src.extractors import ExtractConfig, enrich_inventory_with_text, save_text_outputs

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
inventory_files = sorted(OUTPUT_DIR.glob('inventory_*.parquet'))
assert inventory_files, 'No inventory parquet found. Run 02_inventory.ipynb first.'
INVENTORY_PATH = inventory_files[-1]
print('Using inventory file:', INVENTORY_PATH.name)


Using inventory file: inventory_20260307_074202.parquet


In [2]:
inv = pd.read_parquet(INVENTORY_PATH)
inv = ensure_inventory_schema(inv)
print('Rows:', len(inv))
preview_cols = [c for c in ['relative_path', 'suffix', 'size_bytes'] if c in inv.columns]
display(inv[preview_cols].head(10))
if 'suffix' in inv.columns:
    display(inv['suffix'].fillna('').value_counts().rename_axis('suffix').reset_index(name='count').head(20))
else:
    print('suffix column unavailable after schema backfill')


Rows: 5


,relative_path,suffix,size_bytes
0,downloads/Thumbs.db,.db,0
1,drawings/DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20...,.pdf,14
2,duplicates/copy-a.txt,.txt,14
3,duplicates/copy-b.txt,.txt,14
4,misc/readme.txt,.txt,36


,suffix,count
0,.txt,3
1,.db,1
2,.pdf,1


In [3]:
config = ExtractConfig(
    max_chars_per_file=12000,
    max_csv_rows=30,
    max_csv_columns=20,
    max_xlsx_rows_per_sheet=30,
    max_xlsx_columns=20,
    max_docx_paragraphs=300,
    max_pdf_pages=30,
    preview_chars=300,
)
config


ExtractConfig(max_chars_per_file=12000, max_csv_rows=30, max_csv_columns=20, max_xlsx_rows_per_sheet=30, max_xlsx_columns=20, max_docx_paragraphs=300, max_pdf_pages=30, preview_chars=300)

In [4]:
enriched = enrich_inventory_with_text(inv, path_column='absolute_path', config=config)
display(enriched[['relative_path', 'suffix', 'text_status', 'text_source', 'extracted_chars', 'text_preview']].head(20))


EOF marker not found


,relative_path,suffix,text_status,text_source,extracted_chars,text_preview
0,downloads/Thumbs.db,.db,unsupported,unsupported,0,
1,drawings/DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20...,.pdf,error,pdf,0,
2,duplicates/copy-a.txt,.txt,ok,text,12,same content
3,duplicates/copy-b.txt,.txt,ok,text,12,same content
4,misc/readme.txt,.txt,ok,text,34,Demo text file for inventory scan.


In [5]:
display(enriched['text_status'].value_counts(dropna=False).rename_axis('text_status').reset_index(name='count'))
display(enriched['text_source'].value_counts(dropna=False).rename_axis('text_source').reset_index(name='count'))
display(enriched[enriched['text_status'] == 'error'][['relative_path', 'suffix', 'text_error']].head(20))


,text_status,count
0,ok,3
1,unsupported,1
2,error,1


,text_source,count
0,text,3
1,unsupported,1
2,pdf,1


,relative_path,suffix,text_error
1,drawings/DTC12p500-01_FS_DRWTEC_SITE-LAYOUT_20...,.pdf,PdfStreamError: Stream has ended unexpectedly


In [6]:
display(enriched[enriched['has_extracted_text']][['relative_path', 'text_source', 'extracted_chars', 'text_preview']].head(20))


,relative_path,text_source,extracted_chars,text_preview
2,duplicates/copy-a.txt,text,12,same content
3,duplicates/copy-b.txt,text,12,same content
4,misc/readme.txt,text,34,Demo text file for inventory scan.


In [7]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUT_DIR / f'inventory_with_text_{timestamp}'
csv_path, parquet_path = save_text_outputs(enriched, output_base)
print('Saved CSV   :', csv_path)
print('Saved Parquet:', parquet_path)


Saved CSV   : c:\00_Developement\sch-file-organizer\data\outputs\inventory_with_text_20260307_092059.csv
Saved Parquet: c:\00_Developement\sch-file-organizer\data\outputs\inventory_with_text_20260307_092059.parquet
